6 에이전트 RAG
- AI 에이전트 : 주어진 환경을 관찰하고 판단하며 행동함으로써 특정 목표를 달성하는 AI시스템
- ReAct : 대규모 언어 모델이 추론(Reasoning) + 행동(Acting)을 결합하여 문제를 해결하는 방법론
  . Thought(생각) -> Action(도구 사용) -> Observation(관찰)

In [2]:
import requests
from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    StorageContext,
    load_index_from_storage,
)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.agent import ReActAgent


In [3]:
urls = [
    "https://raw.githubusercontent.com/llama-index-tutorial/llama-index-tutorial/main/ch06/ict_japan_2024.pdf",
    "https://raw.githubusercontent.com/llama-index-tutorial/llama-index-tutorial/main/ch06/ict_usa_2024.pdf"
]

#각 파일 다운로드
for url in urls:
    filename = url.split("/")[-1]
    response = requests.get(url)

    with open(filename, "wb") as f:
        f.write(response.content)
    print(f"{filename} 다운로드 완료")

us_docs = SimpleDirectoryReader(input_files=["ict_usa_2024.pdf"]).load_data()
jp_docs = SimpleDirectoryReader(input_files=["ict_japan_2024.pdf"]).load_data()

print('미국 시장동향 문서의 개수:',len(us_docs))
print('일본 시장동향 문서의 개수:',len(jp_docs))

ict_japan_2024.pdf 다운로드 완료
ict_usa_2024.pdf 다운로드 완료
미국 시장동향 문서의 개수: 29
일본 시장동향 문서의 개수: 30


6.3 허깅페이스 임베딩
- 'BAAI/bge-m3'모델 : 한글 성능이 뛰어난 임베딩 모델

In [4]:
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-m3")

#벡터 인덱스 생성
#문서를 임베딩 모델을 통해 벡터화하여 인덱스 구축
us_index = VectorStoreIndex.from_documents(us_docs, embed_model=embed_model)
jp_index = VectorStoreIndex.from_documents(jp_docs, embed_model=embed_model)

#생성된 벡터 인덱스를 로컬 스토리지에 저장
#이후 재사용시 다시 생성할 필요 없이 저장된 인덱스를 불러올 수 있음
us_index.storage_context.persist(persist_dir="./storage/us")
jp_index.storage_context.persist(persist_dir="./storage/jp")

#벡터 인덱스를 쿼리 엔진으로 변환
us_engine = us_index.as_query_engine(similarity_top_k=5)
jp_engine = jp_index.as_query_engine(similarity_top_k=5)

response = us_engine.query("이 문서의 요약본을 한글로 작성해줘")
print(response)

이 문서는 미국에서의 인공지능(AI) 활용을 통해 혁신과 경쟁을 촉진하고 국가의 리더십을 강화하는 방안에 대해 다루고 있습니다. AI의 책임감 있는 사용을 촉진하고 근로자, 소비자, 환자, 학생 등을 지원하는 내용과 함께, AI의 미국 내 체류와 일할 기회 확대, 국제 표준 개발, 지속가능한 개발 촉진, 정부의 효과적인 AI 활용 등에 대한 내용이 포함되어 있습니다.


In [5]:
#검색 도구 등록
query_engine_tools = [
    QueryEngineTool(
        query_engine=us_engine,
        metadata=ToolMetadata(
            name="usa_ict",
            description=(
                "미국의 ICT 시장동향 정보를 제공합니다. 미국 ICT와 관련된 질문은 해당 도구를 사용하세요."
            ),
        ),
    ),
    QueryEngineTool(
        query_engine=jp_engine,
        metadata=ToolMetadata(
            name="japan_ict",
            description=(
                "일본의 ICT 시장동향 정보를 제공합니다. 일본 ICT와 관련된 질문은 해당 도구를 사용하세요."
            ),
        ),
    ),
]


In [10]:

from llama_index.llms.openai import OpenAI
from llama_index.core.agent import ReActAgent
from llama_index.core.tools import QueryEngineTool

llm = OpenAI(model="gpt-4-0125-preview", temperature=0) 
#llm = OpenAI(model="gpt-4.1", temperature=0)
#에이전트 생성
agent = ReActAgent.from_tools(query_engine_tools, llm=llm, verbose=True)

# 사용
response = agent.chat("일본 시장동향 3줄로 요약해줘.한국어로 대답해야해")
print(response)

> Running step bfb8a399-ad21-43f3-a8c2-bff234bee130. Step input: 일본 시장동향 3줄로 요약해줘.한국어로 대답해야해
Thought: The current language of the user is Korean. I need to use a tool to help me answer the question about the Japanese ICT market trends.
Action: japan_ict
Action Input: {'input': '시장동향 요약'}
Observation: The market trends in Japan include the increasing adoption of biometric authentication for payment systems by various companies such as JR West, Yahoo! Mart, and Seven Bank. These companies are implementing facial recognition technology for ticketing, payment, and ATM transactions to enhance customer experience and streamline services. Additionally, the government is focusing on initiatives like AI learning using public data, promoting digital transformation, and exploring opportunities in the metaverse and NFT development.
> Running step 63e02fd3-1c1f-436a-a6e0-01e491f34624. Step input: None
Thought: I can answer without using any more tools. I'll use the user's language to answer.
Answer

In [11]:
print('기본 프롬프트')
print(agent.get_prompts()['agent_worker:system_prompt'].template)

기본 프롬프트
You are designed to help with a variety of tasks, from answering questions to providing summaries to other types of analyses.

## Tools

You have access to a wide variety of tools. You are responsible for using the tools in any sequence you deem appropriate to complete the task at hand.
This may require breaking the task into subtasks and using different tools to complete each subtask.

You have access to the following tools:
{tool_desc}


## Output Format

Please answer in the same language as the question and use the following format:

```
Thought: The current language of the user is: (user's language). I need to use a tool to help me answer the question.
Action: tool name (one of {tool_names}) if using a tool.
Action Input: the input to the tool, in a JSON format representing the kwargs (e.g. {{"input": "hello world", "num_beams": 5}})
```

Please ALWAYS start with a Thought.

Please use a valid JSON format for the Action Input. Do NOT do this {{'input': 'hello world', 'num_

In [12]:
#프롬프트 재구성
react_system_header_str = """
당신은 질문에 답변하는 것부터 요약 제공, 기타 여러 유형의 분석까지 다양한 작업을 돕기 위해 설계되었습니다.

## 도구
당신은 다양한 도구에 접근할 수 있습니다. 현재 작업을 완료하기 위해 적절하다고 판단되는 순서로 도구를 사용하는 것은 당신의 책임입니다. 이를 위해 작업을 하위 작업으로 나누고, 각하위 작업마다 다른 도구를 사용할 필요가 있을 수 있습니다.

당신은 다음 도구들에 접근할 수 있습니다:
{tool_desc}

## 출력 형식
질문에 답변하기 위해 다음 형식을 사용하십시오.

###
Thought: I need to use a tool to help me answer the question.
Action: 도구 이름 (사용할 도구 중 하나인 {tool_names})
Action Input: 도구에 전달할 입력을 JSON 형식으로 제공합니다. 예: {{\"input\": \"helloworld\", \"num_beams\": 5}}
###

항상 'Thought'로 시작하십시오.

'Action Input'에서는 올바른 JSON 형식을 사용하십시오. 이렇게 쓰지 마십시오: {{\"input\":\"hello world\", \"num_beams\": 5}}.

이 형식이 사용되면, 사용자는 다음 형식으로 응답할 것입니다:

###
Observation: 도구 응답
###

이 형식을 계속 반복하여 더 이상 도구를 사용하지 않고 질문에 답변할 수 있을 만큼 충분한 정보를 얻을 때까지 진행하십시오. 그 시점에서는 반드시 다음 두 가지 형식 중 하나로 응답해야 합니다:

###
Thought: I can answer without using any more tools.
Answer: [여기에 답변을 작성하세요]
###

###
Thought: I cannot answer the question with the provided tools.
Answer: 죄송합니다. 해당 질문에 답변할 수 없습니다.
###

## 추가 규칙
- 답변은 반드시 질문에 도달하기까지의 과정을 설명하는 순차적인 항목들로 구성되어야 합니다.여기에는 인과 관계의 내용이 포함될 수 있습니다.
- 각 도구의 함수 설명을 반드시 준수해야 하며, 함수가 인수를 기대할 경우 인수를 생략하지 마십시오.
- 답변은 항상 한국어로 자세하게 작성되어야 합니다.
- 질문에 대한 답변만 작성하세요. 질문과 관계없는 검색을 수행하지 마십시오. 이는 매우 중요합니다.
- Please follow the thought-action-input format.
- 하나의 질문에 많은 검색어가 포함되어져 있는 것처럼 보인다면 검색어를 나누어서 순차적으로 검색하십시오. 더 좋은 답변을 얻을 수 있을 것입니다.
- 도구를 사용하여 답변할 수 있는 주제라면 반드시 도구를 사용하시기 바랍니다. 이는 매우 중요합니다.
- 당신이 도구 없이 답변하는 것은 도구의 주제와 완전히 다른 주제의 질문이 들어왔을 때 뿐입니다. 도구와 연관된 질문이라면 반드시 도구를 호출하십시오. 이는 매우 중요하며 당신이 지켜야 할 1순위의 우선사항입니다.

## 현재 대화
아래는 인간과 어시스턴트 메시지가 교차되어 있는 현재 대화 내용입니다.
"""


In [15]:
# 1) PromptTemplate 임포트
try:
    from llama_index.core.prompts import PromptTemplate    # 0.10+ 권장
except ImportError:
    try:
        from llama_index.prompts import PromptTemplate     # 구버전 호환
    except ImportError:
        from llama_index import PromptTemplate             # 아주 오래된 버전

react_system_prompt = PromptTemplate(react_system_header_str)
agent.update_prompts({"agent_worker:system_prompt": react_system_prompt})
agent.reset()

response = agent.chat("한국과 미국의 ICT 기관 협력 사례.한국어로 대답해야해")

> Running step 6eaf34b1-7278-4e46-9bbe-f9878e1031a8. Step input: 한국과 미국의 ICT 기관 협력 사례.한국어로 대답해야해
Thought: 한국과 미국의 ICT 기관 협력 사례에 대한 정보를 얻기 위해서는 미국의 ICT 시장동향 정보를 제공하는 도구인 usa_ict를 사용해야 한다. 이 도구를 통해 한국과 미국 간의 ICT 분야에서의 협력 사례에 대한 정보를 얻을 수 있을 것이다.
Action: usa_ict
Action Input: {'input': '한국과 미국의 ICT 기관 협력 사례'}
Observation: Korea and the United States have been noted for their ICT institutional cooperation.
> Running step db1b3083-38ea-4dec-a7ce-c2084846fdde. Step input: None
Thought: I can answer without using any more tools.
Answer: 한국과 미국은 ICT 분야에서 기관 간 협력으로 주목받고 있습니다. 이러한 협력은 양국 간의 기술 교류, 공동 연구 및 개발 프로젝트, 교육 및 훈련 프로그램 등 다양한 형태로 이루어지고 있습니다. 이는 양국의 ICT 산업 발전에 기여하고 있으며, 글로벌 ICT 시장에서의 경쟁력 강화에도 도움을 주고 있습니다.


In [16]:
response = agent.chat("미국과 일본의 ICT 주요 정책의 공통점과 차이점을 설명해줘. 한국어로 댇바해야해")

> Running step 0a6dbbdd-fd49-4d88-9bf1-c26961576b8e. Step input: 미국과 일본의 ICT 주요 정책의 공통점과 차이점을 설명해줘. 한국어로 댇바해야해
Thought: 미국과 일본의 ICT 주요 정책의 공통점과 차이점을 파악하기 위해서는 각각의 최신 ICT 시장 동향과 정책 정보가 필요하다. 먼저 미국의 ICT 정책에 대한 정보를 얻기 위해 usa_ict 도구를 사용하고, 이어서 일본의 ICT 정책에 대한 정보를 얻기 위해 japan_ict 도구를 사용할 것이다.
Action: usa_ict
Action Input: {'input': '미국 ICT 주요 정책'}
Observation: The United States is focusing on enhancing its digital capabilities in various sectors such as defense, drone delivery services, and cybersecurity. The country is actively adopting digital twin technology in defense operations, expanding drone delivery services by companies like Walmart and Amazon, and strengthening cybersecurity measures in response to recent cyberattacks. Additionally, there is a new national cybersecurity strategy in place to protect critical infrastructure and enhance resilience in the digital ecosystem. Major tech companies are also taking steps to improve cloud security measures in light of recent hacking inciden